In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    VotingRegressor,
    StackingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(X.head())
print(X.shape)
print(y.shape)

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  
(442, 10)
(442,)


3. Final Test Set LOCK kar do

Ye 20% test data end tak touch nahi karna.

In [3]:
# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Development data:", X_dev.shape)
print("Final test data:", X_test.shape)

Development data: (353, 10)
Final test data: (89, 10)


In [4]:
import os

# ================================
# CREATE NEW FOLDER
# ================================

os.makedirs("saved_test_data", exist_ok=True)

# ================================
# SAVE TEST DATA
# ================================

X_test.to_csv("saved_test_data/X_test.csv", index=False)
y_test.to_csv("saved_test_data/y_test.csv", index=False)

print("Test data saved successfully!")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Test data saved successfully!
X_test: (89, 10)
y_test: (89,)


# 5. Define 5-Fold Cross Validation

K-Fold will be applied only to the development/training data.

The final test data will remain untouched.

In [6]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Cross Validation created.")

5-Fold Cross Validation created.


# 6. Define Regression Models

Scaling is used inside Pipeline for models where it is useful/important.

Tree-based models do not require StandardScaler.

In [7]:
models = {

    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso())
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor())
    ]),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR())
    ]),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        random_state=42
    )
}

In [8]:
results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        cv=kf,
        scoring={
            "RMSE": "neg_root_mean_squared_error",
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        }
    )

    rmse = -scores["test_RMSE"]
    mae = -scores["test_MAE"]
    r2 = scores["test_R2"]

    results.append({
        "Model": name,
        "Mean RMSE": rmse.mean(),
        "Std RMSE": rmse.std(),
        "Mean MAE": mae.mean(),
        "Mean R2": r2.mean()
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Mean RMSE",
    ascending=True
)

display(results_df)

,Model,Mean RMSE,Std RMSE,Mean MAE,Mean R2
1,Ridge,55.374175,2.373206,44.998879,0.480784
2,Lasso,55.388472,2.324856,45.135762,0.480768
0,Linear Regression,55.394633,2.361496,45.048200,0.480365
3,ElasticNet,56.658537,3.183545,46.866846,0.458352
9,AdaBoost,58.112313,3.974156,48.317252,0.429361
7,Random Forest,59.452977,2.036265,48.481890,0.398955
8,Gradient Boosting,61.109828,2.912421,49.471660,0.367163
4,KNN,62.110398,3.580228,49.442310,0.344657
5,SVR,73.062640,5.526087,61.136210,0.100269
6,Decision Tree,80.410778,9.862003,61.960684,-0.108100


# 8. Select Top 3 Models

We will select the strongest individual models based on cross-validation RMSE.

In [9]:
top_3 = results_df.head(3)

display(top_3)

top_model_names = top_3["Model"].tolist()

print("Top 3 Models:")
for model_name in top_model_names:
    print("-", model_name)

,Model,Mean RMSE,Std RMSE,Mean MAE,Mean R2
1,Ridge,55.374175,2.373206,44.998879,0.480784
2,Lasso,55.388472,2.324856,45.135762,0.480768
0,Linear Regression,55.394633,2.361496,45.048200,0.480365


Top 3 Models:
- Ridge
- Lasso
- Linear Regression


# 9. Hyperparameter Tuning

Hyperparameter tuning will be performed using 5-Fold Cross Validation.

Example: Random Forest.

In [10]:
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_dev, y_dev)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV RMSE:")
print(-grid_search.best_score_)

Best Parameters:
{'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 300}

Best CV RMSE:
58.30802890147832


In [11]:
best_rf = grid_search.best_estimator_

print(best_rf)

Pipeline(steps=[('model',
                 RandomForestRegressor(max_depth=5, min_samples_split=5,
                                       n_estimators=300, random_state=42))])


# 11. Create Other Strong Models for Ensemble

We will combine different strong models.

The goal is to check whether combining models improves performance.

In [12]:
gb_model = GradientBoostingRegressor(
    random_state=42
)

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

# 12. Voting Regression

Voting combines predictions from multiple regression models.

We will evaluate the Voting model using 5-Fold Cross Validation.

In [13]:
voting_model = VotingRegressor([
    ("rf", best_rf),
    ("gb", gb_model),
    ("ridge", ridge_model)
])

voting_scores = cross_validate(
    voting_model,
    X_dev,
    y_dev,
    cv=kf,
    scoring={
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
        "R2": "r2"
    }
)

voting_rmse = -voting_scores["test_RMSE"]
voting_mae = -voting_scores["test_MAE"]
voting_r2 = voting_scores["test_R2"]

print("Voting Mean RMSE:", voting_rmse.mean())
print("Voting Mean MAE :", voting_mae.mean())
print("Voting Mean R2  :", voting_r2.mean())

Voting Mean RMSE: 56.785289744957666
Voting Mean MAE : 46.502400487340424
Voting Mean R2  : 0.4536874807374667


# 13. Stacking Regression

Stacking uses predictions from base models and a meta-model to produce the final prediction.

In [15]:
stacking_model = StackingRegressor(
    estimators=[
        ("rf", best_rf),
        ("gb", gb_model),
        ("ridge", ridge_model)
    ],
    final_estimator=Ridge()
)

stacking_scores = cross_validate(
    stacking_model,
    X_dev,
    y_dev,
    cv=kf,
    scoring={
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error",
        "R2": "r2"
    }
)

stacking_rmse = -stacking_scores["test_RMSE"]
stacking_mae = -stacking_scores["test_MAE"]
stacking_r2 = stacking_scores["test_R2"]

print("Stacking Mean RMSE:", stacking_rmse.mean())
print("Stacking Mean MAE :", stacking_mae.mean())
print("Stacking Mean R2  :", stacking_r2.mean())

Stacking Mean RMSE: 55.363579976309744
Stacking Mean MAE : 45.262873943847225
Stacking Mean R2  : 0.48097590039976856


# 14. Compare Individual Models and Ensemble Models

The lowest RMSE will be considered better.

We should not assume that an ensemble will always beat the best individual model.

In [16]:
ensemble_results = pd.DataFrame({

    "Model": [
        "Best Random Forest",
        "Voting",
        "Stacking"
    ],

    "Mean RMSE": [
        -grid_search.best_score_,
        voting_rmse.mean(),
        stacking_rmse.mean()
    ],

    "Mean MAE": [
        np.nan,
        voting_mae.mean(),
        stacking_mae.mean()
    ],

    "Mean R2": [
        np.nan,
        voting_r2.mean(),
        stacking_r2.mean()
    ]
})

display(
    ensemble_results.sort_values(
        "Mean RMSE"
    )
)

,Model,Mean RMSE,Mean MAE,Mean R2
2,Stacking,55.363580,45.262874,0.480976
1,Voting,56.785290,46.502400,0.453687
0,Best Random Forest,58.308029,NaN,NaN


# 15. Select Final Model

We select the model with the lowest cross-validation RMSE.

The final test set is still untouched.

In [19]:
candidate_models = {
    "Random Forest": best_rf,
    "Voting": voting_model,
    "Stacking": stacking_model
}

candidate_scores = {}

for name, model in candidate_models.items():

    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    candidate_scores[name] = -scores["test_score"].mean()

print(candidate_scores)

best_model_name = min(
    candidate_scores,
    key=candidate_scores.get
)

final_model = candidate_models[best_model_name]

print("\nFINAL MODEL:", best_model_name)

{'Random Forest': np.float64(58.30802890147832), 'Voting': np.float64(56.785289744957666), 'Stacking': np.float64(55.363579976309744)}

FINAL MODEL: Stacking


In [23]:
final_model.fit(X_dev, y_dev)

print("Final model trained.")

Final model trained.


In [24]:
X_test_final = pd.read_csv(
    "saved_test_data/X_test.csv"
)

y_test_final = pd.read_csv(
    "saved_test_data/y_test.csv"
).squeeze()

print("Test X:", X_test_final.shape)
print("Test y:", y_test_final.shape)

Test X: (89, 10)
Test y: (89,)


In [25]:
y_pred = final_model.predict(
    X_test_final
)

print("Predictions generated.")
print(y_pred[:10])

Predictions generated.
[142.01164487 176.97286404 139.69065451 278.49030601 120.41634389
  96.39167007 253.98516266 187.75910453  98.09078474 124.86266439]


In [26]:
final_mae = mean_absolute_error(
    y_test_final,
    y_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test_final,
        y_pred
    )
)

final_r2 = r2_score(
    y_test_final,
    y_pred
)

print("================================")
print("       FINAL TEST RESULTS")
print("================================")

print("MAE  :", final_mae)
print("RMSE :", final_rmse)
print("R²   :", final_r2)

       FINAL TEST RESULTS
MAE  : 42.67955537822213
RMSE : 53.196502494491774
R²   : 0.46587640143776776


In [27]:
comparison = pd.DataFrame({
    "Actual": y_test_final.values,
    "Predicted": y_pred
})

display(comparison.head(20))

,Actual,Predicted
0,219.0,142.011645
1,70.0,176.972864
2,202.0,139.690655
3,230.0,278.490306
4,111.0,120.416344
5,84.0,96.391670
6,242.0,253.985163
7,272.0,187.759105
8,94.0,98.090785
9,96.0,124.862664
